# 🔄 Case Promotion Integration Demo

This notebook demonstrates the **case promotion system** that bridges NLP analytics to operational case management.

## System Overview

The case promotion layer transforms AI risk assessments into actionable cases:

**Data Flow:**
```
Instagram Data → NLP Signals → Risk Profiles → SCS Cases → Youth Workers
                                              ↓
                              Creates/Updates operational cases
                              Tracks risk history over time
                              Auto-generates checklists
```

## Three Database Layers

1. **Analytics Layer** (Input)
   - `instagram_posts`, `instagram_comments`, `instagram_users`
   - `text_units_signals` (NLP outputs)

2. **Risk Assessment Layer** (Intermediate)
   - `case_risk_profiles` (aggregated risk scores)

3. **Operational Layer** (Output) ⭐ **This Demo**
   - `scs_cases` (case records)
   - `scs_case_history` (risk evolution)
   - `scs_checklist` (task tracking)

## Setup

First, ensure you're in the correct directory and import necessary modules.

In [ ]:
import os
import sys
import asyncio
from datetime import datetime
from pprint import pprint
import nest_asyncio

# Enable async/await in Jupyter
nest_asyncio.apply()

# Add backend to path (notebook is in demos/, backend is in parent dir)
repo_root = os.path.dirname(os.getcwd())  # Go up from demos/ to repo root
backend_path = os.path.join(repo_root, 'backend')
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

print(f"✅ Backend path added: {backend_path}")
print(f"📁 Current directory: {os.getcwd()}")
print(f"📁 Repo root: {repo_root}")

# Enable autoawait for top-level await
try:
    get_ipython().run_line_magic('autoawait', 'True')
    print("✅ Async support enabled")
except:
    print("ℹ️  Running in standard Python mode")

✅ Backend path added: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled/demos/backend
📁 Current directory: /Users/pandeymahi/Documents/GitHub/DellInnovate2026_Team-Untitled/demos


## Step 1: Test MongoDB Connection

Verify database connection and check collection status.

In [3]:
from config.database import MongoDB

async def test_connection():
    """Test MongoDB connection and show collection status"""
    try:
        await MongoDB.connect_db()
        db = MongoDB.get_db()
        
        print("=" * 70)
        print("📊 MongoDB Database Status")
        print("=" * 70)
        
        # Analytics layer
        print("\n📈 ANALYTICS LAYER:")
        posts_count = await db.instagram_posts.count_documents({})
        users_count = await db.instagram_users.count_documents({})
        signals_count = await db.text_units_signals.count_documents({})
        print(f"  Instagram posts: {posts_count:,}")
        print(f"  Instagram users: {users_count:,}")
        print(f"  NLP signals: {signals_count:,}")
        
        # Risk assessment layer
        print("\n🎯 RISK ASSESSMENT LAYER:")
        profiles_count = await db.case_risk_profiles.count_documents({})
        print(f"  Risk profiles: {profiles_count:,}")
        
        # Operational layer
        print("\n⚙️  OPERATIONAL LAYER (SCS):")
        cases_count = await db.scs_cases.count_documents({})
        history_count = await db.scs_case_history.count_documents({})
        checklist_count = await db.scs_checklist.count_documents({})
        templates_count = await db.scs_checklist_templates.count_documents({})
        print(f"  Cases: {cases_count:,}")
        print(f"  Case history entries: {history_count:,}")
        print(f"  Checklist items: {checklist_count:,}")
        print(f"  Checklist templates: {templates_count:,}")
        
        print("\n" + "=" * 70)
        
        return db, {
            'profiles': profiles_count,
            'cases': cases_count,
            'history': history_count,
            'checklist': checklist_count,
            'templates': templates_count
        }
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None

# Run the test
db, status = await test_connection()

ModuleNotFoundError: No module named 'config'

## Step 2: Check Risk Profiles (Input for Promotion)

View the risk profiles that will be promoted to cases.

In [ ]:
async def show_risk_profiles(limit=5):
    """Display sample risk profiles"""
    
    profiles = await db.case_risk_profiles.find().limit(limit).to_list(length=limit)
    
    print("=" * 70)
    print(f"📊 Risk Profiles Available ({len(profiles)} shown)")
    print("=" * 70)
    
    for i, profile in enumerate(profiles, 1):
        print(f"\n{i}. User: {profile.get('case_user')}")
        print(f"   Priority: {profile.get('priority_level', 'N/A')}")
        
        # Handle both legacy and PCA scoring
        if 'final_score' in profile:
            print(f"   Risk Score: {profile['final_score']:.3f} (0-1 scale)")
            print(f"   Emotion: {profile.get('emotion_score', 0):.3f}, " 
                  f"Sentiment: {profile.get('sentiment_score', 0):.3f}, "
                  f"Harm: {profile.get('harm_score', 0):.3f}")
        else:
            print(f"   Risk Score: {profile.get('risk_score', 0):.1f}/100")
        
        print(f"   Category: {profile.get('category', 'N/A')}")
        print(f"   Timestamp: {profile.get('timestamp', 'N/A')}")
    
    print("\n" + "=" * 70)
    return profiles

profiles = await show_risk_profiles(limit=5)

📊 Risk Profiles Available (5 shown)

1. User: chrishemsworth
   Priority: low
   Risk Score: 0.000 (0-1 scale)
   Emotion: 0.283, Sentiment: 0.000, Harm: 0.000
   Category: N/A
   Timestamp: 2026-02-28 23:23:23.173000

2. User: crime101film
   Priority: critical
   Risk Score: 1.000 (0-1 scale)
   Emotion: 0.189, Sentiment: 0.000, Harm: 0.000
   Category: N/A
   Timestamp: 2026-02-28 23:24:10.073000

3. User: hollywoodauthentic
   Priority: medium
   Risk Score: 0.407 (0-1 scale)
   Emotion: 0.087, Sentiment: 0.000, Harm: 0.000
   Category: N/A
   Timestamp: 2026-02-28 23:23:23.873000

4. User: markruffalo
   Priority: low
   Risk Score: 0.010 (0-1 scale)
   Emotion: 0.000, Sentiment: 0.000, Harm: 0.000
   Category: N/A
   Timestamp: 2026-02-28 23:35:20.475000

5. User: mrbeast
   Priority: critical
   Risk Score: 1.000 (0-1 scale)
   Emotion: 0.002, Sentiment: 0.088, Harm: 0.000
   Category: N/A
   Timestamp: 2026-02-28 23:23:23.316000



## Step 3: Check/Create Checklist Templates

Ensure checklist templates are available (required for new cases).

In [ ]:
async def check_templates():
    """Check and create default checklist templates if needed"""
    
    templates_count = await db.scs_checklist_templates.count_documents({})
    
    print("=" * 70)
    print("📝 Checklist Templates Status")
    print("=" * 70)
    
    if templates_count == 0:
        print("\n⚠️  No templates found. Creating defaults...\n")
        
        templates = [
            {
                "template_id": 1,
                "label": "Case Analysis Completed",
                "is_mandatory": True,
                "display_order": 1,
                "is_active": True,
                "created_at": datetime.utcnow()
            },
            {
                "template_id": 2,
                "label": "Outreach Attempted",
                "is_mandatory": True,
                "display_order": 2,
                "is_active": True,
                "created_at": datetime.utcnow()
            },
            {
                "template_id": 3,
                "label": "Response Received",
                "is_mandatory": True,
                "display_order": 3,
                "is_active": True,
                "created_at": datetime.utcnow()
            },
            {
                "template_id": 4,
                "label": "Follow-up Scheduled",
                "is_mandatory": True,
                "display_order": 4,
                "is_active": True,
                "created_at": datetime.utcnow()
            }
        ]
        
        for template in templates:
            await db.scs_checklist_templates.update_one(
                {"template_id": template["template_id"]},
                {"$set": template},
                upsert=True
            )
        
        print(f"✅ Created {len(templates)} templates")
    else:
        print(f"\n✅ Found {templates_count} templates")
    
    # Show templates
    templates = await db.scs_checklist_templates.find().sort("display_order", 1).to_list(length=10)
    print("\nAvailable templates:")
    for template in templates:
        mandatory = "✓" if template.get("is_mandatory") else " "
        print(f"  [{mandatory}] {template.get('display_order')}. {template.get('label')}")
    
    print("=" * 70)

await check_templates()

📝 Checklist Templates Status

✅ Found 4 templates

Available templates:
  [✓] 1. Case Analysis Completed
  [✓] 2. Outreach Attempted
  [✓] 3. Response Received
  [✓] 4. Follow-up Scheduled


## Step 4: Run Case Promotion (Main Function)

Promote risk profiles to operational SCS cases. This is the core integration function.

In [ ]:
from services.case_promotion import promote_risk_profiles_to_scs_cases

async def run_promotion(min_priority="low", limit=None):
    """
    Run the case promotion service
    
    Args:
        min_priority: Minimum priority level ('low', 'medium', 'high', 'critical')
        limit: Maximum profiles to process (None = all)
    """
    
    print("=" * 70)
    print("🚀 Running Case Promotion")
    print("=" * 70)
    print(f"Configuration:")
    print(f"  Min Priority: {min_priority}")
    print(f"  Limit: {limit or 'No limit (all profiles)'}")
    print()
    
    # Record counts before
    cases_before = await db.scs_cases.count_documents({})
    history_before = await db.scs_case_history.count_documents({})
    checklist_before = await db.scs_checklist.count_documents({})
    
    # Run promotion
    try:
        results = await promote_risk_profiles_to_scs_cases(
            db=db,
            min_priority=min_priority,
            limit=limit,
            ingestion_timestamp=datetime.utcnow()
        )
        
        print("✅ Promotion Complete!\n")
        print("📊 Results:")
        print(f"  Profiles read: {results['profiles_read']}")
        print(f"  Profiles filtered: {results['profiles_filtered']}")
        print(f"  Cases created: {results['cases_created']} 🆕")
        print(f"  Cases updated: {results['cases_updated']} 🔄")
        print(f"  History entries added: {results['history_entries_added']}")
        print(f"  Checklist items created: {results['checklist_items_created']}")
        
        if results['errors']:
            print(f"\n⚠️  Errors: {len(results['errors'])}")
            for error in results['errors'][:3]:
                print(f"    - {error}")
        
        # Show changes
        cases_after = await db.scs_cases.count_documents({})
        history_after = await db.scs_case_history.count_documents({})
        checklist_after = await db.scs_checklist.count_documents({})
        
        print("\n📈 Database Changes:")
        print(f"  Cases: {cases_before} → {cases_after} (+{cases_after - cases_before})")
        print(f"  History: {history_before} → {history_after} (+{history_after - history_before})")
        print(f"  Checklist: {checklist_before} → {checklist_after} (+{checklist_after - checklist_before})")
        
        print("=" * 70)
        return results
        
    except Exception as e:
        print(f"❌ Promotion failed: {e}")
        import traceback
        traceback.print_exc()
        return None

# Run promotion (start with small limit for testing)
results = await run_promotion(min_priority="low", limit=5)

🚀 Running Case Promotion
Configuration:
  Min Priority: low
  Limit: 5



/var/folders/nl/frj5t53s1ld7f77lqr3vvh000000gn/T/ipykernel_18153/76707521.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingestion_timestamp=datetime.utcnow()
2026-03-07 13:56:07.555 | INFO     | services.case_promotion:promote_profiles_to_cases:51 - Starting case promotion: min_priority=low, limit=5
2026-03-07 13:56:07.625 | INFO     | services.case_promotion:promote_profiles_to_cases:77 - Retrieved 5 risk profiles
2026-03-07 13:56:07.625 | INFO     | services.case_promotion:promote_profiles_to_cases:86 - Filtered to 5 profiles meeting priority threshold
2026-03-07 13:56:07.625 | DEBUG    | services.case_promotion:_process_single_profile:158 - Processing profile for user: astartingpoint
2026-03-07 13:56:07.877 | INFO     | services.case_promotion:_create_new_case:247 - Created case CASE_2026_006 with priority=low, category

✅ Promotion Complete!

📊 Results:
  Profiles read: 5
  Profiles filtered: 5
  Cases created: 5 🆕
  Cases updated: 0 🔄
  History entries added: 5
  Checklist items created: 20

📈 Database Changes:
  Cases: 5 → 10 (+5)
  History: 5 → 10 (+5)
  Checklist: 20 → 40 (+20)


## Step 5: View Created Cases

Display the cases that were just created or updated.

In [ ]:
async def show_cases(priority=None, limit=10):
    """Display cases from scs_cases collection"""
    
    query = {}
    if priority:
        query["priority"] = priority.lower()
    
    cases = await db.scs_cases.find(query).sort("current_risk_score", -1).limit(limit).to_list(length=limit)
    
    print("=" * 70)
    print(f"📋 SCS Cases ({len(cases)} shown)")
    if priority:
        print(f"   Filtered by priority: {priority}")
    print("=" * 70)
    
    for i, case in enumerate(cases, 1):
        print(f"\n{i}. Case ID: {case['case_id']}")
        print(f"   User: {case['user_id']}")
        print(f"   Priority: {case['priority'].upper()} ⭐")
        print(f"   Risk Score: {case['current_risk_score']:.2f}/100")
        print(f"   Category: {case['category']}")
        print(f"   Status: {case['case_status']} / {case['work_status']}")
        print(f"   Assigned To: {case['assigned_to'] or 'Unassigned'}")
        print(f"   Created: {case['created_at']}")
        print(f"   Explanation: {case['ai_explanation'][:100]}...")
    
    print("\n" + "=" * 70)
    return cases

# Show all cases (remove priority filter to see all)
cases = await show_cases(priority=None, limit=10)

📋 SCS Cases (10 shown)

1. Case ID: CASE_2026_009
   User: disneyplus
   Priority: CRITICAL ⭐
   Risk Score: 100.00/100
   Category: Unknown
   Status: unassigned / not_started
   Assigned To: Unassigned
   Created: 2026-03-07 05:56:07.555000
   Explanation: Risk assessment based on aggregated signals: emotion=0.02, sentiment=0.00, harm=0.00. LLM calibratio...

2. Case ID: CASE_2026_001
   User: metsaryhma
   Priority: CRITICAL ⭐
   Risk Score: 100.00/100
   Category: Unknown
   Status: unassigned / not_started
   Assigned To: Unassigned
   Created: 2026-03-07 05:20:11.770000
   Explanation: Risk assessment based on aggregated signals: emotion=0.08, sentiment=0.09, harm=0.00. LLM calibratio...

3. Case ID: CASE_2026_004
   User: mrbeast
   Priority: CRITICAL ⭐
   Risk Score: 100.00/100
   Category: Unknown
   Status: unassigned / not_started
   Assigned To: Unassigned
   Created: 2026-03-07 05:20:11.770000
   Explanation: Risk assessment based on aggregated signals: emotion=0.00, senti

## Step 6: View Case Priority Distribution

See how cases are distributed across priority levels.

In [ ]:
async def show_priority_distribution():
    """Show case distribution by priority"""
    
    pipeline = [
        {"$group": {
            "_id": "$priority",
            "count": {"$sum": 1},
            "avg_risk_score": {"$avg": "$current_risk_score"}
        }},
        {"$sort": {"_id": 1}}
    ]
    
    results = await db.scs_cases.aggregate(pipeline).to_list(length=10)
    
    print("=" * 70)
    print("📊 Case Distribution by Priority")
    print("=" * 70)
    
    priority_order = ["critical", "high", "medium", "low"]
    total = 0
    
    for priority in priority_order:
        found = next((r for r in results if r["_id"] == priority), None)
        if found:
            count = found["count"]
            avg_score = found["avg_risk_score"]
            total += count
            bar = "█" * int(count * 2)  # Visual bar
            print(f"\n{priority.upper():8s}: {count:3d} cases (avg risk: {avg_score:.1f})")
            print(f"          {bar}")
    
    print(f"\nTotal: {total} cases")
    print("=" * 70)

await show_priority_distribution()

📊 Case Distribution by Priority

CRITICAL:   4 cases (avg risk: 98.4)
          ████████

HIGH    :   3 cases (avg risk: 58.3)
          ██████

MEDIUM  :   1 cases (avg risk: 39.0)
          ██

LOW     :   2 cases (avg risk: 0.3)
          ████

Total: 10 cases


## Step 7: View Case History (Risk Evolution)

See how a specific case's risk has changed over time.

In [ ]:
async def show_case_history(case_id=None):
    """Display history for a specific case or pick the first one"""
    
    # If no case_id provided, get the first case
    if not case_id:
        first_case = await db.scs_cases.find_one({})
        if not first_case:
            print("❌ No cases found")
            return
        case_id = first_case['case_id']
    
    # Get case info
    case = await db.scs_cases.find_one({"case_id": case_id})
    if not case:
        print(f"❌ Case {case_id} not found")
        return
    
    # Get history
    history = await db.scs_case_history.find({"case_id": case_id}).sort("ingestion_date", 1).to_list(length=100)
    
    print("=" * 70)
    print(f"📈 Case History: {case_id}")
    print("=" * 70)
    print(f"\nCase Info:")
    print(f"  User: {case['user_id']}")
    print(f"  Current Risk: {case['current_risk_score']:.2f}")
    print(f"  Priority: {case['priority']}")
    print(f"  Category: {case['category']}")
    
    print(f"\n📊 History ({len(history)} entries):")
    print("-" * 70)
    
    for i, entry in enumerate(history, 1):
        print(f"\n{i}. Date: {entry['ingestion_date']}")
        print(f"   Risk Score: {entry['risk_score']:.2f}")
        print(f"   Category: {entry['category']}")
        print(f"   Model: {entry.get('model_version', 'N/A')}")
        print(f"   Explanation: {entry['ai_explanation'][:80]}...")
    
    # Show trend
    if len(history) > 1:
        first_score = history[0]['risk_score']
        last_score = history[-1]['risk_score']
        change = last_score - first_score
        trend = "📈 Increasing" if change > 0 else "📉 Decreasing" if change < 0 else "➡️  Stable"
        print(f"\n{trend} (Change: {change:+.2f})")
    
    print("=" * 70)

# Show history for first case
await show_case_history()

📈 Case History: CASE_2026_001

Case Info:
  User: metsaryhma
  Current Risk: 100.00
  Priority: critical
  Category: Unknown

📊 History (1 entries):
----------------------------------------------------------------------

1. Date: 2026-03-07 05:20:11.770000
   Risk Score: 100.00
   Category: Unknown
   Model: nlp-v1-pca-v1-llm-v1
   Explanation: Risk assessment based on aggregated signals: emotion=0.08, sentiment=0.09, harm=...


## Step 8: View Checklist Items

See the auto-generated checklist items for a case.

In [ ]:
async def show_checklist(case_id=None):
    """Display checklist items for a case"""
    
    # If no case_id provided, get the first case
    if not case_id:
        first_case = await db.scs_cases.find_one({})
        if not first_case:
            print("❌ No cases found")
            return
        case_id = first_case['case_id']
    
    # Get checklist items
    items = await db.scs_checklist.find({"case_id": case_id}).sort("display_order", 1).to_list(length=100)
    
    print("=" * 70)
    print(f"✅ Checklist: {case_id}")
    print("=" * 70)
    
    if not items:
        print("\n⚠️  No checklist items found")
    else:
        for item in items:
            status = "✓" if item['completed'] else "○"
            mandatory = " (required)" if item['is_mandatory'] else ""
            print(f"\n[{status}] {item['label']}{mandatory}")
            print(f"    Created: {item['created_at']}")
            if item['completed']:
                print(f"    Completed: {item['completed_at']} by {item['completed_by']}")
            if item['comments']:
                print(f"    Comments: {len(item['comments'])}")
    
    print("\n" + "=" * 70)

await show_checklist()

✅ Checklist: CASE_2026_001

[○] Case Analysis Completed (required)
    Created: 2026-03-07 05:20:12.471000

[○] Outreach Attempted (required)
    Created: 2026-03-07 05:20:12.541000

[○] Response Received (required)
    Created: 2026-03-07 05:20:12.670000

[○] Follow-up Scheduled (required)
    Created: 2026-03-07 05:20:12.744000



## Step 9: Test Multiple Runs (Update Existing Cases)

Run promotion again to see how existing cases are updated (not recreated).

In [ ]:
async def test_multiple_runs():
    """Test running promotion multiple times"""
    
    print("=" * 70)
    print("🔄 Testing Multiple Promotion Runs")
    print("=" * 70)
    
    # Get initial state
    initial_cases = await db.scs_cases.count_documents({})
    initial_history = await db.scs_case_history.count_documents({})
    initial_checklist = await db.scs_checklist.count_documents({})
    
    print(f"\nInitial State:")
    print(f"  Cases: {initial_cases}")
    print(f"  History: {initial_history}")
    print(f"  Checklist: {initial_checklist}")
    
    # Run promotion twice with same data
    print("\n" + "-" * 70)
    print("Run #1:")
    print("-" * 70)
    results1 = await promote_risk_profiles_to_scs_cases(db=db, min_priority="low", limit=3)
    
    print(f"  Created: {results1['cases_created']}, Updated: {results1['cases_updated']}")
    print(f"  History added: {results1['history_entries_added']}")
    print(f"  Checklist created: {results1['checklist_items_created']}")
    
    print("\n" + "-" * 70)
    print("Run #2 (same data):")
    print("-" * 70)
    results2 = await promote_risk_profiles_to_scs_cases(db=db, min_priority="low", limit=3)
    
    print(f"  Created: {results2['cases_created']}, Updated: {results2['cases_updated']}")
    print(f"  History added: {results2['history_entries_added']}")
    print(f"  Checklist created: {results2['checklist_items_created']}")
    
    # Verify behavior
    final_cases = await db.scs_cases.count_documents({})
    final_history = await db.scs_case_history.count_documents({})
    final_checklist = await db.scs_checklist.count_documents({})
    
    print("\n" + "-" * 70)
    print("Expected Behavior:")
    print("-" * 70)
    print("✅ Run #1: Creates new cases + checklist")
    print("✅ Run #2: Updates existing cases, adds history, NO new checklist")
    
    print(f"\nFinal State:")
    print(f"  Cases: {initial_cases} → {final_cases} (should increase in Run #1 only)")
    print(f"  History: {initial_history} → {final_history} (should increase BOTH runs)")
    print(f"  Checklist: {initial_checklist} → {final_checklist} (should increase in Run #1 only)")
    
    print("\n" + "=" * 70)

await test_multiple_runs()

🔄 Testing Multiple Promotion Runs


2026-03-07 14:02:49.153 | INFO     | services.case_promotion:promote_profiles_to_cases:51 - Starting case promotion: min_priority=low, limit=3
2026-03-07 14:02:49.223 | INFO     | services.case_promotion:promote_profiles_to_cases:77 - Retrieved 3 risk profiles
2026-03-07 14:02:49.223 | INFO     | services.case_promotion:promote_profiles_to_cases:86 - Filtered to 3 profiles meeting priority threshold
2026-03-07 14:02:49.223 | DEBUG    | services.case_promotion:_process_single_profile:158 - Processing profile for user: thinkjinx



Initial State:
  Cases: 10
  History: 10
  Checklist: 40

----------------------------------------------------------------------
Run #1:
----------------------------------------------------------------------


2026-03-07 14:02:49.382 | DEBUG    | services.case_promotion:_update_existing_case:283 - Updated case CASE_2026_003 with new risk data
2026-03-07 14:02:49.541 | DEBUG    | services.case_promotion:_add_case_history:321 - Added history entry for case CASE_2026_003
2026-03-07 14:02:49.542 | INFO     | services.case_promotion:_process_single_profile:190 - Updated existing case CASE_2026_003 for user thinkjinx
2026-03-07 14:02:49.542 | DEBUG    | services.case_promotion:_process_single_profile:158 - Processing profile for user: disneycruiselinesg
2026-03-07 14:02:49.752 | INFO     | services.case_promotion:_create_new_case:247 - Created case CASE_2026_011 with priority=high, category=Unknown
2026-03-07 14:02:49.893 | DEBUG    | services.case_promotion:_add_case_history:321 - Added history entry for case CASE_2026_011
2026-03-07 14:02:50.353 | INFO     | services.case_promotion:_create_checklist_items:362 - Created 4 checklist items for case CASE_2026_011
2026-03-07 14:02:50.354 | INFO     |

  Created: 2, Updated: 1
  History added: 3
  Checklist created: 8

----------------------------------------------------------------------
Run #2 (same data):
----------------------------------------------------------------------


2026-03-07 14:02:51.575 | INFO     | services.case_promotion:_create_new_case:247 - Created case CASE_2026_013 with priority=low, category=Unknown
2026-03-07 14:02:51.729 | DEBUG    | services.case_promotion:_add_case_history:321 - Added history entry for case CASE_2026_013
2026-03-07 14:02:52.240 | INFO     | services.case_promotion:_create_checklist_items:362 - Created 4 checklist items for case CASE_2026_013
2026-03-07 14:02:52.241 | INFO     | services.case_promotion:_process_single_profile:174 - Created new case CASE_2026_013 for user mileycyrus
2026-03-07 14:02:52.242 | DEBUG    | services.case_promotion:_process_single_profile:158 - Processing profile for user: taylorswift
2026-03-07 14:02:52.379 | DEBUG    | services.case_promotion:_update_existing_case:283 - Updated case CASE_2026_005 with new risk data
2026-03-07 14:02:52.519 | DEBUG    | services.case_promotion:_add_case_history:321 - Added history entry for case CASE_2026_005
2026-03-07 14:02:52.520 | INFO     | services.ca

  Created: 2, Updated: 1
  History added: 3
  Checklist created: 8

----------------------------------------------------------------------
Expected Behavior:
----------------------------------------------------------------------
✅ Run #1: Creates new cases + checklist
✅ Run #2: Updates existing cases, adds history, NO new checklist

Final State:
  Cases: 10 → 14 (should increase in Run #1 only)
  History: 10 → 16 (should increase BOTH runs)
  Checklist: 40 → 56 (should increase in Run #1 only)



## Step 10: Query Cases by Different Criteria

Test various query patterns that the dashboard might use.

In [ ]:
async def query_examples():
    """Show various query examples"""
    
    print("=" * 70)
    print("🔍 Query Examples")
    print("=" * 70)
    
    # 1. Unassigned critical cases
    print("\n1️⃣  Unassigned Critical Cases:")
    critical_unassigned = await db.scs_cases.find({
        "priority": "critical",
        "case_status": "unassigned"
    }).sort("current_risk_score", -1).to_list(length=5)
    
    print(f"   Found: {len(critical_unassigned)}")
    for case in critical_unassigned:
        print(f"   - {case['case_id']}: {case['user_id']} (risk: {case['current_risk_score']:.1f})")
    
    # 2. Cases by category
    print("\n2️⃣  Cases by Category (Depression):")
    depression_cases = await db.scs_cases.find({
        "category": {"$regex": "depression", "$options": "i"}
    }).limit(3).to_list(length=3)
    
    print(f"   Found: {len(depression_cases)}")
    for case in depression_cases:
        print(f"   - {case['case_id']}: {case['user_id']}")
    
    # 3. High risk cases (score > 70)
    print("\n3️⃣  High Risk Cases (Score > 70):")
    high_risk = await db.scs_cases.find({
        "current_risk_score": {"$gt": 70}
    }).sort("current_risk_score", -1).limit(5).to_list(length=5)
    
    print(f"   Found: {len(high_risk)}")
    for case in high_risk:
        print(f"   - {case['case_id']}: Score {case['current_risk_score']:.1f}, Priority: {case['priority']}")
    
    # 4. Recent cases (last 24 hours)
    from datetime import timedelta
    yesterday = datetime.utcnow() - timedelta(days=1)
    
    print("\n4️⃣  Cases Created in Last 24 Hours:")
    recent = await db.scs_cases.find({
        "created_at": {"$gte": yesterday}
    }).to_list(length=100)
    
    print(f"   Found: {len(recent)}")
    
    # 5. Cases with history count
    print("\n5️⃣  Cases with Multiple History Entries:")
    pipeline = [
        {"$lookup": {
            "from": "scs_case_history",
            "localField": "case_id",
            "foreignField": "case_id",
            "as": "history"
        }},
        {"$addFields": {"history_count": {"$size": "$history"}}},
        {"$match": {"history_count": {"$gt": 1}}},
        {"$sort": {"history_count": -1}},
        {"$limit": 3}
    ]
    
    cases_with_history = await db.scs_cases.aggregate(pipeline).to_list(length=3)
    print(f"   Found: {len(cases_with_history)}")
    for case in cases_with_history:
        print(f"   - {case['case_id']}: {case['history_count']} history entries")
    
    print("\n" + "=" * 70)

await query_examples()

🔍 Query Examples

1️⃣  Unassigned Critical Cases:
   Found: 4
   - CASE_2026_001: metsaryhma (risk: 100.0)
   - CASE_2026_004: mrbeast (risk: 100.0)
   - CASE_2026_009: disneyplus (risk: 100.0)
   - CASE_2026_007: mohammed.usrof (risk: 93.5)

2️⃣  Cases by Category (Depression):
   Found: 0

3️⃣  High Risk Cases (Score > 70):
   Found: 4
   - CASE_2026_001: Score 100.0, Priority: critical
   - CASE_2026_004: Score 100.0, Priority: critical
   - CASE_2026_009: Score 100.0, Priority: critical
   - CASE_2026_007: Score 93.5, Priority: critical

4️⃣  Cases Created in Last 24 Hours:
   Found: 14

5️⃣  Cases with Multiple History Entries:
   Found: 2
   - CASE_2026_003: 2 history entries
   - CASE_2026_005: 2 history entries



/var/folders/nl/frj5t53s1ld7f77lqr3vvh000000gn/T/ipykernel_18153/4151378048.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  yesterday = datetime.utcnow() - timedelta(days=1)


## Summary & Next Steps

### ✅ What We Tested

1. ✅ MongoDB connection and collection status
2. ✅ Risk profiles availability (input data)
3. ✅ Checklist template setup
4. ✅ Case promotion execution
5. ✅ Created cases viewing
6. ✅ Priority distribution
7. ✅ Case history tracking
8. ✅ Checklist items display
9. ✅ Multiple run behavior (upsert logic)
10. ✅ Various query patterns

### 💡 Key Findings

- **New cases**: Get case ID, history entry, and 4 checklist items
- **Existing cases**: Updated risk fields, new history entry, checklist preserved
- **History tracking**: Grows with each ingestion cycle
- **Priority filtering**: Configurable thresholds work correctly

### 🚀 Next Steps

1. **Production Use**: Run the automated scheduler
   ```bash
   cd backend
   python pipeline_scheduler.py
   ```

2. **API Integration**: Test via REST endpoints
   ```bash
   curl -X POST http://localhost:8000/api/analytics/promote-to-cases
   ```

3. **Frontend Connection**: Connect dashboard to `scs_cases` collection

4. **Monitoring**: Track pipeline execution via `pipeline_execution_log`

### 📚 Documentation

- [CASE_PROMOTION_README.md](backend/CASE_PROMOTION_README.md) - Quick start
- [CASE_PROMOTION_INTEGRATION.md](backend/CASE_PROMOTION_INTEGRATION.md) - Full docs
- [CASE_PROMOTION_QUICKREF.md](backend/CASE_PROMOTION_QUICKREF.md) - Commands

## Cleanup (Optional)

Uncomment and run the cell below if you want to reset the test data.

In [ ]:
# # ⚠️  WARNING: This will delete all SCS operational data!
# # Uncomment only if you want to reset for testing

# async def cleanup_test_data():
#     """Remove test cases (use with caution!)"""
    
#     print("⚠️  Cleaning up test data...")
    
#     result_cases = await db.scs_cases.delete_many({})
#     result_history = await db.scs_case_history.delete_many({})
#     result_checklist = await db.scs_checklist.delete_many({})
    
#     print(f"Deleted {result_cases.deleted_count} cases")
#     print(f"Deleted {result_history.deleted_count} history entries")
#     print(f"Deleted {result_checklist.deleted_count} checklist items")
    
#     print("✅ Cleanup complete")

# # Uncomment to run:
# await cleanup_test_data()

# #print("ℹ️  Cleanup cell is commented out. Uncomment to reset test data.")

⚠️  Cleaning up test data...
Deleted 14 cases
Deleted 16 history entries
Deleted 56 checklist items
✅ Cleanup complete
